## Problem 1: Heat Pump, Carnot Machine

Ideal Carnot heat pump between $T_c = 263$ K and $T_h = 308$ K.

A heat pump is a refrigerator run backwards. What you want is heat delivered to the
hot side, so the COP is defined with $Q_h$ on top, not $Q_c$:

$$\text{COP}_{hp} = \frac{Q_h}{W} = \frac{T_h}{T_h - T_c}$$

Energy balance per cycle: $Q_h = Q_c + W$.

In [1]:
Tc = 263.0
Th = 308.0
Qh = 10.0

COP = Th / (Th - Tc)
W = Qh / COP
Qc = Qh - W

print("1) COP =", round(COP, 3))
print("2) W   =", round(W, 3), "kW")
print("3) Qc  =", round(Qc, 3), "kW")

1) COP = 6.844
2) W   = 1.461 kW
3) Qc  = 8.539 kW


## Problem 2: R410a-based machine

$T_c = 253$ K $= -20\ ^\circ$C, $T_h = 313$ K $= 40\ ^\circ$C. Both are labelled
isotherms on the chart, so no interpolation is needed.

Isentropic compression and expansion plus isothermal evaporation and condensation
means this is a Carnot cycle, run inside the two-phase dome. Four states:

| State | Where on the chart | Process leaving it |
|---|---|---|
| 1 | saturated vapour, 40 °C (right edge of dome, 24 bar) | 1→2 condensation at $T_h$, heat out |
| 2 | saturated liquid, 40 °C (left edge of dome) | 2→3 isentropic expansion down to 4 bar |
| 3 | 4 bar, same $s$ as state 2 | 3→4 evaporation at $T_c$, heat in |
| 4 | 4 bar, same $s$ as state 1 | 4→1 isentropic compression back to 24 bar |

States 3 and 4 sit inside the dome, so read them off the quality lines:

$$x = \frac{s - s_f}{s_g - s_f}, \qquad h = h_f + x\,(h_g - h_f)$$

Then:

$$Q_h = h_1 - h_2, \qquad Q_c = h_4 - h_3, \qquad W = Q_h - Q_c, \qquad \text{COP} = \frac{Q_h}{W}$$

In [ ]:
Tc, Th = 253.0, 313.0

h1, s1 = 425.0, 1.729
h2, s2 = 266.0, 1.221
hf, sf = 170.0, 0.889
hg, sg = 414.0, 1.852

x3 = (s2 - sf) / (sg - sf)
x4 = (s1 - sf) / (sg - sf)
h3 = hf + x3 * (hg - hf)
h4 = hf + x4 * (hg - hf)

Qh = h1 - h2
Qc = h4 - h3
W = Qh - Qc
COP = Qh / W

m_dot = 10.0 / Qh

print("x3 =", round(x3, 3), "  h3 =", round(h3, 1), "kJ/kg")
print("x4 =", round(x4, 3), "  h4 =", round(h4, 1), "kJ/kg")
print("1) COP =", round(COP, 2), "  (Carnot limit:", round(Th / (Th - Tc), 2), ")")
print("2) Qh =", round(Qh, 1), "kJ/kg   Qc =", round(Qc, 1), "kJ/kg   W =", round(W, 1), "kJ/kg")
print("3) mass flow =", round(m_dot * 1000, 1), "g/s")

The COP lands on the Carnot value $T_h/(T_h - T_c) = 5.22$, which is the check that
the chart was read correctly. Any Carnot cycle gives the same COP no matter what
fluid runs through it. What the refrigerant decides is the size of the machine:
R410a moves 159 kJ per kilogram, so 63 g/s of it is enough for 10 kW.

Real heat pumps score lower, around 3 to 4 at these temperatures. They throttle
through a valve instead of expanding isentropically, which wastes the expansion work
and is irreversible.

## Problem 3: SCOP

$$\text{SCOP} = \frac{\text{total annual heat out}}{\text{total annual electricity in}}$$

Each row gets its own curve, picked by the supply temperature $T_h$:

| Row | $T_h$ | Curve | $T_c$ (outdoor) | COP read off |
|---|---|---|---|---|
| T < 0 | 55 °C | W55, bottom | -3 °C | 2.6 |
| 0 < T < 5 | 45 °C | W45, middle | 2 °C | 3.2 |
| 5 < T < 10 | 35 °C | W35, top | 7 °C | 4.4 |

Per row: $\text{heat} = P_{heat} \times \text{hours}$ and $\text{electricity} = \text{heat} / \text{COP}$.
Then sum both columns and divide.

In [2]:
rows = [
    ("T < 0", 55, -3, 6, 1000, 2.6),
    ("0 < T < 5", 45, 2, 4, 2000, 3.2),
    ("5 < T < 10", 35, 7, 2, 3000, 4.4),
]

total_heat = 0.0
total_elec = 0.0

print(f"{'range':12}{'Th':>4}{'Tc':>4}{'P':>4}{'hours':>7}{'heat kWh':>10}{'COP':>6}{'elec kWh':>10}")
for name, Th, Tc, P, hours, COP in rows:
    heat = P * hours
    elec = heat / COP
    total_heat += heat
    total_elec += elec
    print(f"{name:12}{Th:>4}{Tc:>4}{P:>4}{hours:>7}{heat:>10.0f}{COP:>6}{elec:>10.0f}")

SCOP = total_heat / total_elec

print(f"\n{'TOTAL':12}{'':4}{'':4}{'':4}{'':7}{total_heat:>10.0f}{'':6}{total_elec:>10.0f}")
print("SCOP =", round(SCOP, 2))

range         Th  Tc   P  hours  heat kWh   COP  elec kWh
T < 0         55  -3   6   1000      6000   2.6      2308
0 < T < 5     45   2   4   2000      8000   3.2      2500
5 < T < 10    35   7   2   3000      6000   4.4      1364

TOTAL                               20000            6171
SCOP = 3.24


SCOP (3.24) sits below the plain average of the three COP values (3.4). It has to.
The weighting is by electricity used, and the cold band burns the most electricity
per kWh of heat, so the worst COP carries the most weight.

That cold band is only 1000 hours out of 6000, but it eats 37% of the yearly
electricity. This is why manufacturers push for low supply temperatures:
underfloor heating at 35 °C instead of old radiators at 55 °C would move every row
up to the top curve.

## Problem 4: Heat Pump vs Gas Stove

Follow 1 kWh of natural gas down both paths.

**Gas stove:** burns it on the spot at 100%, so you get 1 kWh of heat.

**Heat pump:** the gas is burned at a power plant with efficiency $\eta$, 90% of that
electricity survives the grid, then the pump multiplies it by the SCOP:

$$Q_{heat} = \eta \times 0.90 \times 3.5$$

Set the two equal and solve for the plant efficiency that makes them tie:

$$\eta \times 0.90 \times 3.5 = 1 \quad \Rightarrow \quad \eta = \frac{1}{0.90 \times 3.5}$$

In [3]:
SCOP = 3.5
grid = 0.90
eta_stove = 1.00

eta_break = eta_stove / (SCOP * grid)
print("1) break even plant efficiency =", round(eta_break * 100, 1), "%")
print()

for label, eta in [("simple cycle gas turbine", 0.38),
                   ("fleet average CCGT", 0.45),
                   ("modern CCGT", 0.60)]:
    heat = eta * grid * SCOP
    print(f"{label:26} eta={eta:.2f} -> {heat:.2f} kWh heat per kWh gas")
print(f"{'gas stove':26} eta=1.00 -> 1.00 kWh heat per kWh gas")

1) break even plant efficiency = 31.7 %

simple cycle gas turbine   eta=0.38 -> 1.20 kWh heat per kWh gas
fleet average CCGT         eta=0.45 -> 1.42 kWh heat per kWh gas
modern CCGT                eta=0.60 -> 1.89 kWh heat per kWh gas
gas stove                  eta=1.00 -> 1.00 kWh heat per kWh gas


### 2) Typical natural gas plant efficiency

Simple cycle gas turbine: 33% to 43%. Combined cycle (CCGT), where the hot exhaust
drives a second steam turbine: around 60%, with the best units near 64%.

Even the worst of these clears the 31.7% break even, so the heat pump delivers more
heat per unit of gas. With a modern CCGT it delivers almost twice as much.

### 3) Why no thermal plant reaches 100%

A power plant is a heat engine. The second law says a heat engine cannot turn all of
its heat into work, because the working fluid has to be returned to its starting
state, and that means dumping heat into the surroundings. The ceiling is Carnot:

$$\eta_{max} = 1 - \frac{T_c}{T_h}$$

A gas turbine runs at maybe 1700 K inlet and rejects to roughly 300 K ambient, so
even a perfect, frictionless version tops out near 82%.

$T_h$ cannot just be raised to close that gap. Turbine blades melt. That temperature
limit is set by materials, not by thermodynamics.

Real plants land well under the Carnot number because every step leaks: combustion
itself is irreversible, heat crosses finite temperature gaps in the boiler and
condenser, gas rubs and swirls through the turbine, and the generator and pumps take
their cut.

Note that the 100% gas stove does not break this rule. It never converts heat to
work. It is heat in, heat out. Converting heat into work is the expensive direction,
and it is exactly what the power plant has to do.